In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

df = pd.read_csv("../data/raw/dataset.csv")
display(df.head())

,id,text,relations,diagnosis,solutions,id_1,label_1,start_offset_1,end_offset_1,id_2,label_2,start_offset_2,end_offset_2,id_3,label_3,start_offset_3,end_offset_3
0,249,A cybersquatting domain save-russia[.]today is...,"[{'from_id': 44658, 'id': 9, 'to_id': 44659, '...",The diagnosis is a cyber attack that involves ...,1. Implementing DNS filtering to block access ...,44656,attack-pattern,2,16,44657,url,24,43,44658.0,attack-pattern,57.0,68.0
1,14309,"Like the Android Maikspy, it first sends a not...","[{'from_id': 48531, 'id': 445, 'to_id': 48532,...",The diagnosis is that the entity identified as...,1. Implementing a robust anti-malware software...,48530,SOFTWARE,9,17,48531,malware,17,24,48532.0,Infrastucture,63.0,73.0
2,13996,While analyzing the technical details of this ...,"[{'from_id': 48781, 'id': 461, 'to_id': 48782,...",Diagnosis: APT37/Reaper/Group 123 is responsib...,1. Implementing advanced threat detection tech...,48781,threat-actor,188,194,48782,threat-actor,210,217,48783.0,threat-actor,220.0,229.0
3,13600,(Note that Flash has been declared end-of-life...,"[{'from_id': 51688, 'id': 1133, 'to_id': 51689...",The diagnosis is a malware infection. The enti...,1. Implementing a robust antivirus software th...,51687,TIME,62,79,51688,malware,207,215,51689.0,malware,247.0,258.0
4,14364,Figure 21. Connection of Maikspy variants to 1...,"[{'from_id': 51780, 'id': 1161, 'to_id': 44372...",The diagnosis is that Maikspy malware variants...,1. Implementing a robust firewall system that ...,51779,URL,163,191,51777,URL,70,93,51781.0,malware,120.0,127.0


In [2]:
cols_to_drop = [
    "id", "relations", "diagnosis", "solutions", 
    "id_1", "id_2", "id_3", 
    "start_offset_1", "end_offset_1", 
    "start_offset_2", "end_offset_2", 
    "start_offset_3", "end_offset_3"
]
df = df.drop(columns=cols_to_drop, errors='ignore')
display(df.head())

,text,label_1,label_2,label_3
0,A cybersquatting domain save-russia[.]today is...,attack-pattern,url,attack-pattern
1,"Like the Android Maikspy, it first sends a not...",SOFTWARE,malware,Infrastucture
2,While analyzing the technical details of this ...,threat-actor,threat-actor,threat-actor
3,(Note that Flash has been declared end-of-life...,TIME,malware,malware
4,Figure 21. Connection of Maikspy variants to 1...,URL,URL,malware


In [3]:
# 1. Normalização (ignorando NaNs temporariamente para não dar erro de string)
for col in ["label_1", "label_2", "label_3"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower()

df = df[df['label_3'] != 'email'] 

# 2. Agrupamento em lista
df['labels_list'] = df[['label_1', 'label_2', 'label_3']].values.tolist()

# 3. Limpeza (remove NaNs, espaços e duplicatas) e criação da 'labels_cleaned'
df['labels_cleaned'] = df['labels_list'].apply(
    lambda x: list(set([str(i).lower() for i in x if str(i).lower() != 'nan' and str(i).strip() != '']))
)


# 4. Removemos as colunas que não precisamos mais (com segurança)
df = df.drop(columns=['label_1', 'label_2', 'label_3', 'labels_list'], errors='ignore')



display(df[['text', 'labels_cleaned']].head())

,text,labels_cleaned
0,A cybersquatting domain save-russia[.]today is...,"[attack-pattern, url]"
1,"Like the Android Maikspy, it first sends a not...","[software, malware, infrastucture]"
2,While analyzing the technical details of this ...,[threat-actor]
3,(Note that Flash has been declared end-of-life...,"[malware, time]"
4,Figure 21. Connection of Maikspy variants to 1...,"[malware, url]"


In [4]:
from sklearn.preprocessing import MultiLabelBinarizer
import json

mlb = MultiLabelBinarizer()
encoded_data = mlb.fit_transform(df['labels_cleaned'])

# Salvar o mapeamento para usar na inferência/modelo
label_map = {str(i): label for i, label in enumerate(mlb.classes_)}

# Garante que a pasta existe antes de salvar
os.makedirs('../data/processed', exist_ok=True)
with open('../data/processed/label_map.json', 'w') as f:
    json.dump(label_map, f)

# IMPORTANTE: Converter para float32. O BCEWithLogitsLoss do PyTorch exige floats.
one_hot_df = pd.DataFrame(encoded_data.astype(float), columns=mlb.classes_)

df['labels'] = one_hot_df.values.tolist()

final_df = df[['text', 'labels']].copy()


display(final_df.head())

,text,labels
0,A cybersquatting domain save-russia[.]today is...,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"Like the Android Maikspy, it first sends a not...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, ..."
2,While analyzing the technical details of this ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,(Note that Flash has been declared end-of-life...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
4,Figure 21. Connection of Maikspy variants to 1...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."


In [5]:
print(final_df['labels'])

0      [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
1      [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, ...
2      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
3      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...
4      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...
                             ...                        
468    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...
469    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...
470    [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...
471    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
472    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ...
Name: labels, Length: 473, dtype: object


In [6]:
from datasets import Dataset
from utils import preprocess_function
from datasets import load_from_disk

# 1. Cria o dataset HF
hf_dataset = Dataset.from_pandas(final_df)

# 2. Mapeia a tokenização e REMOVE a coluna 'text'
hf_dataset = hf_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=['text']
)

# 3. Salva no disco
dataset_path = '../data/processed/dataset_hf_processado'
hf_dataset.save_to_disk(dataset_path)

# 4. Verificação imediata
ds = load_from_disk(dataset_path)

print("Chaves do dataset:", ds[0].keys()) 
print("Tipo do input_id:", type(ds[0]['input_ids'][0])) 
print("Exemplo de labels:", ds[0]['labels'])

/home/giulia-mezaroba/Faculdade-Codigos/Cyber-Threat-Dataset-with-Transformers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████| 473/473 [00:00<00:00, 71672.90 examples/s]

Chaves do dataset: dict_keys(['labels', 'input_ids', 'attention_mask'])
Tipo do input_id: <class 'int'>
Exemplo de labels: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
